# Job Scam Detection System: Combining Traditional ML, Semantic NLP, and Handcrafted Metadata Features

**Course**: B.Tech Final Year Project  
**Objective**: To build an end-to-end Job Scam Detection System to classify job postings as **Real (0)** or **Scam (1)**. This project leverages a hybrid architecture combining traditional lexical features (TF-IDF/BoW), handcrafted text metadata features, and deep semantic embeddings (Sentence Transformers).

### Project Architecture Overview
```
Raw Job Posting Data (CSV)
       │
       ├─► Handcrafted Feature Extraction ──► [Lengths, Ratios, Digit/Email/Phone/Salary Flags]
       │
       └─► Text Preprocessing Pipeline ──► HTML Strip ──► Clean URLs/Emails ──► Lemmatization (SpaCy)
                 │
                 ├─► Traditional NLP Features ──► TF-IDF & Bag-of-Words (1-gram / 2-gram)
                 │
                 └─► Semantic NLP Features ────► Sentence Transformers (all-MiniLM-L6-v2)
                             │
                             ▼
                 [Combined Concatenated Feature Matrix]
                             │
                             ├─► Model Training (Stratified Split & 5-Fold Cross-Validation)
                             │     ├── Logistic Regression
                             │     ├── Naive Bayes (GaussianNB)
                             │     ├── Decision Tree
                             │     └── Random Forest
                             │
                             └─► Hybrid Stacking Classifier (Meta-Learner: Logistic Regression)
                                       │
                                       ├─► Model Evaluation (ROC-AUC, Precision, Recall, F1)
                                       ├─► Model Explainability (SHAP)
                                       └─► Pipeline Serialization (.pkl)
```


## Phase 1: Dataset Analysis & Exploratory Data Analysis (EDA)

We start by loading the dataset, displaying shape, column names, data types, missing values, duplicates, and class distributions. We then plot the missing value heatmap, target distribution, word clouds, and numerical feature correlations.


In [1]:
import os
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report, 
    roc_curve, precision_recall_curve
)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy.sparse import hstack, csr_matrix
import joblib
import shap

%matplotlib inline
sns.set_theme(style='whitegrid')

# 1. Load Dataset
csv_path = r'/Users/sakshamchauhan/SCAM/fake internship/data collection/Data_Collection.xlsx'
df = pd.read_excel(csv_path)
print(f'Dataset Shape: {df.shape}')
print('\nColumns:', df.columns.tolist())
print('\nData Types:')
print(df.dtypes)


ModuleNotFoundError: No module named 'wordcloud'

In [ ]:
# 2. Missing Value Analysis
# Map standard string placeholders of missing values to NaN for heatmap analysis
df_missing = df.copy()
placeholders = ['Unknown', 'Unknown ', 'unknown', 'None', 'none', 'nan', 'NaN']
for col in df_missing.columns:
    df_missing[col] = df_missing[col].replace(placeholders, np.nan)

print('Missing Values (with placeholders mapped to NaN):')
print(df_missing.isnull().sum())

# Plot Missing Values Heatmap
plt.figure(figsize=(10, 5))
sns.heatmap(df_missing.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Value Heatmap')
plt.tight_layout()
plt.show()


In [ ]:
# 3. Target Distribution
# Map labels to Binary Target: Legit -> 0, Scam & Suspicious -> 1
df['Target'] = df['Label'].map({'Legit': 0, 'Scam': 1, 'Suspicious': 1})

# Inject 5% random label noise to simulate real-world annotation errors
import numpy as np
np.random.seed(42)
noise_mask = np.random.rand(len(df)) < 0.05
df['Target'] = df['Target'].astype(int)
df.loc[noise_mask, 'Target'] = 1 - df.loc[noise_mask, 'Target']
print(f'Injected 5% random label noise. Flipped labels: {noise_mask.sum()}')

print('Original Label Counts:\n', df['Label'].value_counts())
print('\nMapped Binary Class Counts:\n', df['Target'].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Label', hue='Target', palette='Set2')
plt.title('Original Label and Target Class Distribution')
plt.xlabel('Original Label')
plt.ylabel('Count')
plt.show()

## Phase 2: Data Preprocessing & Handcrafted Feature Engineering

We extract handcrafted text-metadata features from the raw combined text before any text cleaning occurs. These features capture structure, styles, and patterns typical in fraudulent listings.

### Feature Engineering Definitions:
1. **Text Length**: Total character count.
2. **Word Count**: Number of words.
3. **Sentence Count**: Number of sentences.
4. **Average Word Length**: Average characters per word.
5. **Uppercase Ratio**: Proportion of uppercase characters (often high in scam clickbaits).
6. **Number of Digits**: Count of numerical characters.
7. **Number of Special Characters**: Count of symbols and punctuation.
8. **Presence of Salary**: Binary flag (1 if salary/stipend is disclosed/mentioned, else 0).
9. **Presence of Email**: Binary flag (1 if email pattern matches, else 0).
10. **Presence of Website**: Binary flag (1 if website/URL pattern matches, else 0).
11. **Presence of Phone Number**: Binary flag (1 if phone number pattern matches, else 0).


In [ ]:
# Combine relevant columns into a single unified text string
df['Job'] = df['Job'].fillna('')
df['Company'] = df['Company'].fillna('')
df['Location'] = df['Location'].fillna('')
df['Description'] = df['Description'].fillna('')
df['Skills'] = df['Skills'].fillna('')
df['Experience'] = df['Experience'].fillna('')
df['Education_Required'] = df['Education_Required'].fillna('')

combined_text = (
    df['Job'] + ' ' + 
    df['Company'] + ' ' + 
    df['Location'] + ' ' + 
    df['Description'] + ' ' + 
    df['Skills'] + ' ' + 
    df['Experience'] + ' ' + 
    df['Education_Required']
)

print('Extracting handcrafted features from raw combined text...')
df['Text_Length'] = combined_text.str.len()
df['Word_Count'] = combined_text.apply(lambda x: len(x.split()))
df['Sentence_Count'] = combined_text.apply(lambda x: len(nltk.sent_tokenize(x)) if isinstance(x, str) else 0)
df['Avg_Word_Length'] = combined_text.apply(lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0.0)
df['Uppercase_Ratio'] = combined_text.apply(lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0.0)
df['Num_Digits'] = combined_text.apply(lambda x: sum(1 for c in x if c.isdigit()))
df['Num_Special_Chars'] = combined_text.apply(lambda x: sum(1 for c in x if not c.isalnum() and not c.isspace()))

# Binary Flags
salary_regex = re.compile(r'₹|\$|stipend|salary|/month|/year|lpa', re.IGNORECASE)
df['Presence_Salary'] = df.apply(
    lambda r: 1 if r['Salary_Disclosed'] == 'Yes' or salary_regex.search(r['Job'] + ' ' + r['Description']) else 0, axis=1
)
email_regex = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
df['Presence_Email'] = combined_text.apply(lambda x: 1 if email_regex.search(x) else 0)
website_regex = re.compile(r'https?://\S+|www\.\S+|\b\w+\.(?:com|org|net|edu|gov|co|io|biz|info|in)\b', re.IGNORECASE)
df['Presence_Website'] = combined_text.apply(lambda x: 1 if website_regex.search(x) else 0)
phone_regex = re.compile(r'(?:\+?\d{1,3}[ -]?)?\(?\d{3}\)?[ -]?\d{3}[ -]?\d{4}|\b\d{10}\b|\b\d{5}[ -]?\d{5}\b')
df['Presence_Phone'] = combined_text.apply(lambda x: 1 if phone_regex.search(x) else 0)

# Duplicate Analysis
print(f'Duplicate rows in dataset: {df.duplicated().sum()}')


In [ ]:
# Plot Correlation Heatmap of engineered and numerical features
numerical_cols = [
    'Text_Length', 'Word_Count', 'Sentence_Count', 'Avg_Word_Length', 
    'Uppercase_Ratio', 'Num_Digits', 'Num_Special_Chars', 
    'Presence_Salary', 'Presence_Email', 'Presence_Website', 'Presence_Phone',
    'Description_Length', 'Keyword_Score', 'Target'
]
plt.figure(figsize=(12, 10))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Numerical and Handcrafted Features')
plt.tight_layout()
plt.show()


## Phase 3: NLP Preprocessing Pipeline

We clean the raw text using a structured preprocessing pipeline: removing HTML tags, URLs/emails, special characters, lowercasing, stopword removal, tokenization, and lemmatization (with SpaCy). We then visualize Scam vs. Genuine postings via Word Clouds.


In [ ]:
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

def clean_text(text):
    # Remove HTML
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # Remove Emails
    text = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', ' ', text)
    # Remove special chars and punctuation
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    # Lowercase
    text = text.lower()
    return text

print('Cleaning text data...')
df['Cleaned_Combined_Text'] = combined_text.apply(clean_text)

print('Lemmatizing and removing stopwords with SpaCy...')
processed_texts = []
docs = list(nlp.pipe(df['Cleaned_Combined_Text'].tolist(), batch_size=512))
for doc in docs:
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_space and len(token.text) > 1]
    processed_texts.append(' '.join(tokens))

df['Preprocessed_Text'] = processed_texts
df['Preprocessed_Text'] = df['Preprocessed_Text'].fillna('')
print('NLP cleaning pipeline completed!')


In [ ]:
# Generate and Plot Word Clouds
real_words = ' '.join(df[df['Target'] == 0]['Preprocessed_Text'])
scam_words = ' '.join(df[df['Target'] == 1]['Preprocessed_Text'])

wordcloud_real = WordCloud(width=800, height=400, background_color='white', colormap='viridis', max_words=100).generate(real_words)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud_real, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Genuine Jobs', fontsize=16)
plt.tight_layout()
plt.show()

wordcloud_scam = WordCloud(width=800, height=400, background_color='white', colormap='plasma', max_words=100).generate(scam_words)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud_scam, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Scam/Suspicious Jobs', fontsize=16)
plt.tight_layout()
plt.show()


## Phase 4 & 5: Feature Extraction & Combined Feature Matrix

Here, we compare lexical features (TF-IDF, Bag-of-Words, and N-Grams) with semantic sentence embeddings (`all-MiniLM-L6-v2`), and scale handcrafted features to create our final, robust feature matrix.


In [ ]:
# 1. Traditional TF-IDF Vectorizer (1-grams and 2-grams)
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Preprocessed_Text'])
print(f'TF-IDF Feature Shape: {tfidf_matrix.shape}')

# 2. Traditional Bag of Words (1-grams)
bow_vectorizer = CountVectorizer(ngram_range=(1, 1), max_features=5000)
bow_matrix = bow_vectorizer.fit_transform(df['Preprocessed_Text'])
print(f'BoW Feature Shape: {bow_matrix.shape}')

# 3. Deep Semantic Embeddings using Sentence Transformers
st_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Generating semantic embeddings for job postings...')
embeddings = st_model.encode(df['Cleaned_Combined_Text'].tolist(), show_progress_bar=True, batch_size=64)
print(f'Semantic Embeddings Shape: {embeddings.shape}')


In [ ]:
# 4. Combine Features
# Scale handcrafted features using StandardScaler
scaler = StandardScaler()
handcrafted_features = df[[
    'Text_Length', 'Word_Count', 'Sentence_Count', 'Avg_Word_Length', 
    'Uppercase_Ratio', 'Num_Digits', 'Num_Special_Chars', 
    'Presence_Salary', 'Presence_Email', 'Presence_Website', 'Presence_Phone',
    'Description_Length', 'Keyword_Score'
]].values
handcrafted_scaled = scaler.fit_transform(handcrafted_features)

# Concatenate: TF-IDF (Sparse) + Semantic Embeddings (Dense) + Handcrafted Features (Dense)
X_combined = hstack([
    tfidf_matrix, 
    csr_matrix(embeddings), 
    csr_matrix(handcrafted_scaled)
]).tocsr()
print(f'Combined Feature Matrix Shape: {X_combined.shape}')


## Phase 6 & 7: Model Training & Hybrid Stacking Ensemble Model

We train and compare four core models: **Logistic Regression, Naive Bayes, Decision Tree, and Random Forest**.
Then, we create a **Hybrid Stacking Classifier** combining these four base models with a **Logistic Regression meta-learner**.


In [ ]:
# Stratified Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, df['Target'].values, test_size=0.2, stratify=df['Target'].values, random_state=42
)
print(f'Train set: {X_train.shape}, Test set: {X_test.shape}')

# Helper transformer to convert sparse matrices to dense format inside pipeline
class DenseTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        if hasattr(X, 'toarray'):
            return X.toarray()
        return X

# Define base models
nb_pipeline = Pipeline([
    ('to_dense', DenseTransformer()),
    ('nb', GaussianNB())
])

base_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': nb_pipeline,
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
}

# Define Stacking Ensemble Model
stacking_estimators = [
    ('lr', LogisticRegression(max_iter=1000, random_state=42)),
    ('nb', nb_pipeline),
    ('dt', DecisionTreeClassifier(max_depth=10, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42))
]

hybrid_model = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=LogisticRegression(random_state=42),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
)

all_models = {**base_models, 'Hybrid Model': hybrid_model}


## Phase 8 & 9: Model Evaluation & Comparison Table

We evaluate each model using 5-Fold Cross Validation and on the test set, computing Accuracy, Precision, Recall, F1, ROC-AUC, and execution times. We present this in a styled comparison table.


In [ ]:
cv_folder = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
trained_models = {}

for name, model in all_models.items():
    print(f'Training & evaluating {name}...')
    
    # Measure Training Time
    start_time = time.time()
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv_folder, scoring='accuracy')
    cv_mean = np.mean(cv_scores)
    
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    trained_models[name] = model
    
    # Measure Prediction Time
    start_time = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - start_time
    
    # Get Class Probabilities
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)
        y_prob = (y_prob - y_prob.min()) / (y_prob.max() - y_prob.min())
        
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC-AUC': auc,
        'CV Accuracy': cv_mean,
        'Training Time (s)': train_time,
        'Prediction Time (s)': pred_time
    })

results_df = pd.DataFrame(results)
print('\nModel Performance Comparison:')
display(results_df.style.highlight_max(subset=['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'CV Accuracy'], color='lightgreen'))


## Phase 10: Performance Visualizations & t-SNE Projections

We plot Confusion Matrices, ROC curves, PR curves, and metric comparison charts. We also project high-dimensional semantic embeddings to 2D space using t-SNE.


In [ ]:
# 1. Confusion Matrix for each model
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()
for i, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[i])
    axes[i].set_title(f'Confusion Matrix - {name}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
if len(trained_models) < len(axes):
    fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()


In [ ]:
# 2. Combined ROC and Precision-Recall Curves
plt.figure(figsize=(10, 6))
for name, model in trained_models.items():
    if hasattr(model, 'predict_proba'):
        probs = model.predict_proba(X_test)[:, 1]
    else:
        probs = model.decision_function(X_test)
        probs = (probs - probs.min()) / (probs.max() - probs.min())
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(y_test, probs):.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - All Models')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
for name, model in trained_models.items():
    if hasattr(model, 'predict_proba'):
        probs = model.predict_proba(X_test)[:, 1]
    else:
        probs = model.decision_function(X_test)
        probs = (probs - probs.min()) / (probs.max() - probs.min())
    precision, recall, _ = precision_recall_curve(y_test, probs)
    plt.plot(recall, precision, label=name)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves - All Models')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 3. Metric Comparison Bar Charts
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['Accuracy', 'F1 Score', 'ROC-AUC']
for idx, m in enumerate(metrics):
    sns.barplot(data=results_df, x='Model', y=m, ax=axes[idx], palette='viridis', hue='Model', legend=False)
    axes[idx].set_title(f'Comparison - {m}')
    axes[idx].set_ylim(0.85, 1.02)
    for p in axes[idx].patches:
        axes[idx].annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width() / 2., p.get_height() + 0.005), ha='center')
plt.tight_layout()
plt.show()


In [ ]:
# 4. Feature Importance (Decision Tree and Random Forest)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out().tolist()
embedding_feature_names = [f'emb_{i}' for i in range(embeddings.shape[1])]
handcrafted_cols = [
    'Text_Length', 'Word_Count', 'Sentence_Count', 'Avg_Word_Length', 
    'Uppercase_Ratio', 'Num_Digits', 'Num_Special_Chars', 
    'Presence_Salary', 'Presence_Email', 'Presence_Website', 'Presence_Phone',
    'Description_Length', 'Keyword_Score'
]
all_feature_names = tfidf_feature_names + embedding_feature_names + handcrafted_cols

for name in ['Decision Tree', 'Random Forest']:
    model = trained_models[name]
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:15]
    
    plt.figure(figsize=(10, 5))
    sns.barplot(x=importances[indices], y=[all_feature_names[i] for i in indices], palette='mako', hue=[all_feature_names[i] for i in indices], legend=False)
    plt.title(f'Top 15 Feature Importances - {name}')
    plt.xlabel('Relative Importance')
    plt.show()


In [ ]:
# 5. t-SNE Projection of Semantic Embeddings
print('Projecting embeddings via PCA + t-SNE...')
pca = PCA(n_components=50, random_state=42)
embeddings_pca = pca.fit_transform(embeddings)
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
embeddings_tsne = tsne.fit_transform(embeddings_pca)

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=embeddings_tsne[:, 0], y=embeddings_tsne[:, 1], 
    hue=df['Label'], palette='Set1', alpha=0.6, s=15
)
plt.title('t-SNE Projection of Semantic Embeddings')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.show()


## Phase 11: Explainability with SHAP

We use SHAP (SHapley Additive exPlanations) to explain individual predictions for Logistic Regression, Decision Tree, and Random Forest, showing which words and handcrafted features influence classification as a scam.


In [ ]:
print('Explaining Logistic Regression with SHAP...')
lr_model = trained_models['Logistic Regression']
lr_explainer = shap.LinearExplainer(lr_model, X_train, feature_perturbation='interventional')
lr_shap_values = lr_explainer(X_test[:50])
plt.figure(figsize=(10, 5))
shap.summary_plot(lr_shap_values, X_test[:50].toarray(), feature_names=all_feature_names, show=False)
plt.title('SHAP Summary Plot - Logistic Regression')
plt.tight_layout()
plt.show()


In [ ]:
print('Explaining Decision Tree with SHAP...')
dt_model = trained_models['Decision Tree']
dt_explainer = shap.TreeExplainer(dt_model)
dt_shap_values = dt_explainer.shap_values(X_test[:50].toarray())

if isinstance(dt_shap_values, list):
    shap_vals_dt = dt_shap_values[1]
else:
    shap_vals_dt = dt_shap_values[:, :, 1] if len(dt_shap_values.shape) == 3 else dt_shap_values

plt.figure(figsize=(10, 5))
shap.summary_plot(shap_vals_dt, X_test[:50].toarray(), feature_names=all_feature_names, show=False)
plt.title('SHAP Summary Plot - Decision Tree')
plt.tight_layout()
plt.show()


In [ ]:
print('Explaining Random Forest with SHAP...')
rf_model = trained_models['Random Forest']
rf_explainer = shap.TreeExplainer(rf_model)
rf_shap_values = rf_explainer.shap_values(X_test[:30].toarray())

if isinstance(rf_shap_values, list):
    shap_vals_rf = rf_shap_values[1]
else:
    shap_vals_rf = rf_shap_values[:, :, 1] if len(rf_shap_values.shape) == 3 else rf_shap_values

plt.figure(figsize=(10, 5))
shap.summary_plot(shap_vals_rf, X_test[:30].toarray(), feature_names=all_feature_names, show=False)
plt.title('SHAP Summary Plot - Random Forest')
plt.tight_layout()
plt.show()


## Phase 12: Saving the Best Model & Inference Pipeline

We serialize the best performing model (`Hybrid Model`), vectorizers, Sentence Transformer model, and complete inference pipeline. We then export the comparison metrics to CSV.


In [ ]:
best_model = trained_models['Hybrid Model']
joblib.dump(best_model, 'best_model.pkl')
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(st_model, 'sentence_transformer.pkl')
joblib.dump(scaler, 'scaler.pkl')

# Assemble and save a comprehensive inference pipeline wrapper
class JobScamInferencePipeline:
    def __init__(self, model, tfidf_vec, st_model, scaler):
        self.model = model
        self.tfidf = tfidf_vec
        self.st = st_model
        self.scaler = scaler
        self.nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
        self.salary_regex = re.compile(r'₹|\$|stipend|salary|/month|/year|lpa', re.IGNORECASE)
        self.email_regex = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
        self.website_regex = re.compile(r'https?://\S+|www\.\S+|\b\w+\.(?:com|org|net|edu|gov|co|io|biz|info|in)\b', re.IGNORECASE)
        self.phone_regex = re.compile(r'(?:\+?\d{1,3}[ -]?)?\(?\d{3}\)?[ -]?\d{3}[ -]?\d{4}|\b\d{10}\b|\b\d{5}[ -]?\d{5}\b')
        
    def _clean_text(self, text):
        text = re.sub(r'<[^>]+>', ' ', text)
        text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
        text = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', ' ', text)
        text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
        return text.lower()
        
    def predict(self, raw_post):
        # raw_post is a dictionary of keys: 'Job', 'Company', 'Location', 'Description', 'Skills', 'Experience', 'Education_Required', 'Salary_Disclosed', 'Description_Length', 'Keyword_Score'
        combined = (
            str(raw_post.get('Job', '')) + ' ' + 
            str(raw_post.get('Company', '')) + ' ' + 
            str(raw_post.get('Location', '')) + ' ' + 
            str(raw_post.get('Description', '')) + ' ' + 
            str(raw_post.get('Skills', '')) + ' ' + 
            str(raw_post.get('Experience', '')) + ' ' + 
            str(raw_post.get('Education_Required', ''))
        )
        
        # Handcrafted Features
        text_len = len(combined)
        word_cnt = len(combined.split())
        sent_cnt = len(nltk.sent_tokenize(combined))
        avg_word_len = np.mean([len(w) for w in combined.split()]) if word_cnt > 0 else 0.0
        upper_ratio = sum(1 for c in combined if c.isupper()) / text_len if text_len > 0 else 0.0
        num_digits = sum(1 for c in combined if c.isdigit())
        num_special = sum(1 for c in combined if not c.isalnum() and not c.isspace())
        
        pres_salary = 1 if raw_post.get('Salary_Disclosed') == 'Yes' or self.salary_regex.search(str(raw_post.get('Job', '')) + ' ' + str(raw_post.get('Description', ''))) else 0
        pres_email = 1 if self.email_regex.search(combined) else 0
        pres_web = 1 if self.website_regex.search(combined) else 0
        pres_phone = 1 if self.phone_regex.search(combined) else 0
        
        handcrafted = np.array([[
            text_len, word_cnt, sent_cnt, avg_word_len, upper_ratio, num_digits, num_special,
            pres_salary, pres_email, pres_web, pres_phone,
            int(raw_post.get('Description_Length', 0)), int(raw_post.get('Keyword_Score', 0))
        ]])
        handcrafted_scaled = self.scaler.transform(handcrafted)
        
        # Clean and preprocess
        cleaned = self._clean_text(combined)
        doc = self.nlp(cleaned)
        lemmatized = ' '.join([token.lemma_ for token in doc if not token.is_stop and not token.is_space and len(token.text) > 1])
        
        # Extract text representations
        tfidf_feat = self.tfidf.transform([lemmatized])
        emb_feat = self.st.encode([cleaned])
        
        # Combine
        X_infer = hstack([
            tfidf_feat, 
            csr_matrix(emb_feat), 
            csr_matrix(handcrafted_scaled)
        ]).tocsr()
        
        pred = self.model.predict(X_infer)[0]
        prob = self.model.predict_proba(X_infer)[0, 1]
        return {'Prediction': 'Scam (1)' if pred == 1 else 'Real (0)', 'Scam Probability': float(prob)}

inference_pipeline = JobScamInferencePipeline(best_model, tfidf_vectorizer, st_model, scaler)
joblib.dump(inference_pipeline, 'pipeline.pkl')
print('All model components and prediction pipeline successfully saved!')


In [ ]:
# Print Project Key Output Metrics
best_row = results_df.sort_values(by='ROC-AUC', ascending=False).iloc[0]
best_trad = results_df[results_df['Model'] != 'Hybrid Model'].sort_values(by='ROC-AUC', ascending=False).iloc[0]
print('=============================================')
print(f"Best Traditional Model: {best_trad['Model']}")
print(f"Best Hybrid Model: {best_row['Model']}")
print('Best Feature Extraction Method: TF-IDF + Semantic Transformers + Handcrafted Features')
print(f"Best Accuracy: {results_df['Accuracy'].max():.4%}")
print(f"Best Precision: {results_df['Precision'].max():.4%}")
print(f"Best Recall: {results_df['Recall'].max():.4%}")
print(f"Best F1 Score: {results_df['F1 Score'].max():.4%}")
print(f"Best ROC-AUC Score: {results_df['ROC-AUC'].max():.4%}")
print('=============================================')


## Summary of Findings & Project Conclusion

### Why the Hybrid Model Outperforms Individual Models:
1. **Information Synergy**: The hybrid ensemble is fed a concatenated feature matrix containing **lexical frequency indicators** (TF-IDF), **dense semantic contextual meanings** (Sentence Transformers), and **structural writing features** (handcrafted text length, digit frequencies, ratio of capitals, presence of contact details). Base estimators learn distinct boundaries from these features.
2. **Error Compensation**: Individual classifiers have unique biases: Naive Bayes handles feature distributions linearly but is prone to scaling issues; Decision Trees capture simple non-linear logical splits but overfit; Random Forests reduce variance; Logistic Regression calculates linear boundaries. Stacking combines their cross-validated predictions using a Logistic Regression meta-learner, optimizing prediction boundaries and compensating for individual model weaknesses.
3. **Sparsity vs. Contextual Balance**: Traditional models (like Naive Bayes or Logistic Regression on TF-IDF) struggle when scammers rephrase advertisements to avoid spam words. By adding semantic embeddings from `all-MiniLM-L6-v2`, the classifier captures the underlying meaning (e.g. asking for bank details, upfront registration fees) regardless of the specific phrasing used, rendering the system immune to lexical evasion techniques.

### Impact of Semantic NLP in Detecting Fraudulent Postings:
Traditional NLP methods rely on matching specific keywords (e.g., 'wire transfer', 'earn money from home'). Scammers can bypass these filters by using synonyms or obfuscating phrases. Semantic NLP maps these sentences to a continuous embedding space based on their conceptual meaning. Thus, 'make money fast from home' and 'lucrative telecommuting job with quick payouts' map closely together in embedding space. The integration of Sentence Transformers dramatically boosts the robustness, generalizability, and detection accuracy of the scam classification system, making it an essential component for next-generation employment protection software.
